In [7]:

# EXPERIMENT: Clean, Integrate and Transform Electronic Healthcare Records

# STEP 1: Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import seaborn as sns
from pylab import rcParams

from pandas import read_excel, read_csv
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from google.colab import files

print("Libraries imported successfully!")


# STEP 2: Upload EHR Datasets
print("\nUpload Patient Information and Medical Conditions CSV files")

uploaded = files.upload()
file_names = list(uploaded.keys())

if len(file_names) < 2:
    raise ValueError("Please upload both EHR CSV files.")


# STEP 3: Automatically Identify Both Datasets
datasets = []

for filename in file_names:
    data = pd.read_csv(filename)
    data.columns = data.columns.str.strip().str.upper()
    datasets.append(data)

# Identify Patient Information Dataset
patient_index = next(
    (i for i, data in enumerate(datasets)
     if 'ID' in data.columns and
     ('BIRTHDATE' in data.columns or 'GENDER' in data.columns)),
    None
)

# Identify Medical Conditions Dataset
condition_index = next(
    (i for i, data in enumerate(datasets)
     if 'PATIENT' in data.columns and
     'START' in data.columns and
     'STOP' in data.columns),
    None
)

if patient_index is None or condition_index is None:
    raise ValueError(
        "Could not identify both datasets. "
        "Please upload patient information and medical conditions CSV files."
    )

dataframe = datasets[patient_index]
dataframe1 = datasets[condition_index]

print("\n========== PATIENT INFORMATION ==========")
display(dataframe.head())
print("Patient Information Shape:", dataframe.shape)

print("\n========== MEDICAL CONDITIONS ==========")
display(dataframe1.head())
print("Medical Conditions Shape:", dataframe1.shape)


# STEP 4: Rename Patient ID Columns
dataframe = dataframe.rename(columns={'ID': 'PATIENTID'})
dataframe1 = dataframe1.rename(columns={'PATIENT': 'PATIENTID'})

print("\n========== AFTER RENAMING PATIENT ID ==========")
print("Patient Information Columns:")
print(dataframe.columns.tolist())

print("\nMedical Conditions Columns:")
print(dataframe1.columns.tolist())


# STEP 5: Display Data Types
print("\n========== PATIENT INFORMATION DATA TYPES ==========")
print(dataframe.dtypes)

print("\n========== MEDICAL CONDITIONS DATA TYPES ==========")
print(dataframe1.dtypes)


# STEP 6: CLEANING - Remove Unnecessary Columns
dataframe = dataframe.drop(
    columns=['DRIVERS', 'SUFFIX'],
    errors='ignore'
)

print("\n========== AFTER REMOVING UNNECESSARY COLUMNS ==========")
display(dataframe.head())


# STEP 7: CLEANING - Remove Duplicate Records
print("\nDuplicate Patient Records Before Cleaning:",
      dataframe.duplicated().sum())

print("Duplicate Condition Records Before Cleaning:",
      dataframe1.duplicated().sum())

dataframe = dataframe.drop_duplicates()
dataframe1 = dataframe1.drop_duplicates()

print("\n========== AFTER REMOVING DUPLICATES ==========")
print("Patient Records:", len(dataframe))
print("Condition Records:", len(dataframe1))
print("Remaining Patient Duplicates:", dataframe.duplicated().sum())
print("Remaining Condition Duplicates:", dataframe1.duplicated().sum())


# STEP 8: CLEANING - Remove Completely Empty Columns
dataframe = dataframe.dropna(axis=1, how='all')
dataframe1 = dataframe1.dropna(axis=1, how='all')

print("\n========== AFTER REMOVING EMPTY COLUMNS ==========")
print("Patient Information Shape:", dataframe.shape)
print("Medical Conditions Shape:", dataframe1.shape)


# STEP 9: INTEGRATION - Merge Both Datasets
df = pd.merge(
    dataframe,
    dataframe1,
    on='PATIENTID',
    how='inner'
)

print("\n========== INTEGRATED HEALTHCARE DATASET ==========")
display(df.head())
print("Integrated Dataset Shape:", df.shape)


# STEP 10: Display Integrated Dataset Data Types
print("\n========== INTEGRATED DATASET DATA TYPES ==========")
print(df.dtypes)


# STEP 11: TRANSFORMATION - Convert Date Columns
df['START'] = pd.to_datetime(df['START'], errors='coerce')
df['STOP'] = pd.to_datetime(df['STOP'], errors='coerce')

print("\n========== AFTER DATE CONVERSION ==========")
display(df[['START', 'STOP']].head())

print("\nUpdated Data Types:")
print(df[['START', 'STOP']].dtypes)


# STEP 12: TRANSFORMATION - Calculate Healthcare Coverage Length
df['HEALTHCARE_COVERAGE_LENGTH'] = (
    df['STOP'] - df['START']
).dt.days

print("\n========== HEALTHCARE COVERAGE LENGTH ==========")
display(
    df[['START', 'STOP', 'HEALTHCARE_COVERAGE_LENGTH']].head()
)


# STEP 13: TRANSFORMATION - Convert Gender Values
df['GENDER'] = df['GENDER'].replace({
    'M': 'Male',
    'F': 'Female'
})

print("\n========== AFTER GENDER TRANSFORMATION ==========")
display(df[['GENDER']].head())


# STEP 14: Display Final Data Types
print("\n========== FINAL DATA TYPES ==========")
print(df.dtypes)


# STEP 15: Display Final Healthcare Records
print("\n========== FINAL CLEANED HEALTHCARE DATASET ==========")
display(df.head())

print("\nFinal Dataset Shape:", df.shape)


# STEP 16: Save Final Dataset
df.to_csv('cleaned_records.csv', index=False)

print("\nClean, Integrate and Transform Operations Completed Successfully!")
print("Output File: cleaned_records.csv")


# STEP 17: Download Final Dataset
files.download('cleaned_records.csv')


Libraries imported successfully!

Upload Patient Information and Medical Conditions CSV files


Saving conditions (1).csv to conditions (1) (2).csv
Saving patients.csv to patients (2).csv

========== PATIENT INFORMATION ==========


,ID,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,...,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
0,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,2019-02-17,NaN,999-65-3251,NaN,NaN,NaN,Damon455,Langosh790,NaN,...,Middleborough Massachusetts US,620 Lynch Tunnel Apt 0,Springfield,Massachusetts,Hampden County,1104.0,42.080389,-72.480431,9039.1645,7964.1255
1,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaN,999-49-3323,S99941126,NaN,NaN,Thi53,Wunsch504,NaN,...,Danvers Massachusetts US,972 Tillman Branch Suite 48,Bellingham,Massachusetts,Norfolk County,NaN,42.035213,-71.482519,402723.4150,14064.1350
2,339144f8-50e1-633e-a013-f361391c4cff,1998-05-11,NaN,999-10-8743,S99996708,X75063318X,Mr.,Chi716,Greenfelder433,NaN,...,Athens Athens Prefecture GR,1060 Bernhard Crossroad Suite 15,Boston,Massachusetts,Suffolk County,2131.0,42.292557,-71.061160,571935.8725,787.5375
3,d488232e-bf14-4bed-08c0-a82f34b6a197,2003-01-28,NaN,999-56-6057,S99929424,NaN,Ms.,Phillis443,Walter473,NaN,...,Boston Massachusetts US,677 Ritchie Terrace,Hingham,Massachusetts,Plymouth County,2043.0,42.200491,-70.916076,582557.8030,104782.2070
4,217f95a3-4e10-bd5d-fb67-0cfb5e8ba075,1993-12-23,NaN,999-91-4320,S99991143,X44132498X,Mr.,Jerrold404,Herzog843,NaN,...,Boston Massachusetts US,276 Bernier Branch,Revere,Massachusetts,Suffolk County,NaN,42.381875,-70.999286,475826.8550,18067.0950


Patient Information Shape: (1163, 25)

========== MEDICAL CONDITIONS ==========


,START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION
0,2013-06-24,2013-07-02,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,0b2794bd-ec2b-d34f-0610-2523b3b7fcf0,10509002,Acute bronchitis (disorder)
1,2016-02-27,2016-03-14,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,a6d818dd-0983-fd1c-eefa-3d2295532c45,283371005,Laceration of forearm
2,2016-08-11,2016-08-22,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,36d2e781-4655-0a11-1f70-c69856e02019,444814009,Viral sinusitis (disorder)
3,2016-11-27,2016-12-17,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,c8eaaf41-958b-31ab-7de5-568cee8751f3,444814009,Viral sinusitis (disorder)
4,2017-02-22,2017-06-02,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,6474f606-5a1b-48c0-bbbf-ad6dcbc24d4e,16114001,Fracture of ankle


Medical Conditions Shape: (38094, 6)

========== AFTER RENAMING PATIENT ID ==========
Patient Information Columns:
['PATIENTID', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE']

Medical Conditions Columns:
['START', 'STOP', 'PATIENTID', 'ENCOUNTER', 'CODE', 'DESCRIPTION']

========== PATIENT INFORMATION DATA TYPES ==========
PATIENTID               object
BIRTHDATE               object
DEATHDATE               object
SSN                     object
DRIVERS                 object
PASSPORT                object
PREFIX                  object
FIRST                   object
LAST                    object
SUFFIX                  object
MAIDEN                  object
MARITAL                 object
RACE                    object
ETHNICITY               object
GENDER                 

... (10 outputs left out)

,PATIENTID,BIRTHDATE,DEATHDATE,SSN,PASSPORT,PREFIX,FIRST,LAST,MAIDEN,MARITAL,...,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,START,STOP,ENCOUNTER,CODE,DESCRIPTION,HEALTHCARE_COVERAGE_LENGTH
0,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaN,999-49-3323,NaN,NaN,Thi53,Wunsch504,NaN,NaN,...,42.035213,-71.482519,402723.415,14064.135,2013-06-24,2013-07-02,0b2794bd-ec2b-d34f-0610-2523b3b7fcf0,10509002,Acute bronchitis (disorder),8.0
1,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaN,999-49-3323,NaN,NaN,Thi53,Wunsch504,NaN,NaN,...,42.035213,-71.482519,402723.415,14064.135,2016-02-27,2016-03-14,a6d818dd-0983-fd1c-eefa-3d2295532c45,283371005,Laceration of forearm,16.0
2,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaN,999-49-3323,NaN,NaN,Thi53,Wunsch504,NaN,NaN,...,42.035213,-71.482519,402723.415,14064.135,2016-08-11,2016-08-22,36d2e781-4655-0a11-1f70-c69856e02019,444814009,Viral sinusitis (disorder),11.0
3,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaN,999-49-3323,NaN,NaN,Thi53,Wunsch504,NaN,NaN,...,42.035213,-71.482519,402723.415,14064.135,2016-11-27,2016-12-17,c8eaaf41-958b-31ab-7de5-568cee8751f3,444814009,Viral sinusitis (disorder),20.0
4,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaN,999-49-3323,NaN,NaN,Thi53,Wunsch504,NaN,NaN,...,42.035213,-71.482519,402723.415,14064.135,2017-02-22,2017-06-02,6474f606-5a1b-48c0-bbbf-ad6dcbc24d4e,16114001,Fracture of ankle,100.0



Final Dataset Shape: (38094, 29)

Clean, Integrate and Transform Operations Completed Successfully!
Output File: cleaned_records.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>